In [ ]:
from copy import copy
from random import shuffle

import albumentations as A
import cv2
import ipyplot
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms as T
from albumentations.pytorch import ToTensorV2
from imutils.perspective import order_points, four_point_transform
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, TensorDataset

import notebook_setup  # isort:skip
from acid.torch.setup import BoardModelSetup  # isort:skip
from acid.utils.cv2 import resize_and_pad  # isort:skip

In [ ]:
all_images = list(BoardModelSetup.dataset_path.rglob("*.jp*"))
shuffle(all_images)
IMAGE_TO_TEST = list(all_images)[0]
IMAGE_WORK_SIZE = (2048, 1024)

In [ ]:
model = BoardModelSetup.model
model.eval()

# prepare images
img = Image.open(IMAGE_TO_TEST)
img_orig = copy(img)
img_orig = ImageOps.pad(img_orig, IMAGE_WORK_SIZE)
img_orig = np.asarray(img_orig)

img = ImageOps.pad(img, BoardModelSetup.image_size)
img = BoardModelSetup.transforms["val"](image=np.asarray(img))["image"]

In [ ]:
with torch.no_grad():
    pred = model(img.unsqueeze(0))

mask = pred[0].detach().numpy()

mask[mask > 0] = 0
mask[mask < 0] = 1

In [ ]:
plt.imshow(mask[0, :, :])
plt.show()

gray = cv2.cvtColor(mask[0, :, :], cv2.COLOR_GRAY2BGR)
gray = (255 / gray.max() * (gray - gray.min())).astype(np.uint8)
_, gray = cv2.threshold(gray, 110, 1, cv2.THRESH_BINARY)

# resize back to working size
gray = resize_and_pad(gray, IMAGE_WORK_SIZE)

contours, _ = cv2.findContours(gray[:, :, 0], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour = max(contours, key=cv2.contourArea)
contour_len = cv2.arcLength(contour, True)
board_edges = cv2.approxPolyDP(contour, 0.05 * contour_len, True)
board_edges = board_edges.reshape(-1, 2)
board_edges = order_points(board_edges)

cv2.drawContours(gray, contours, -1, (255), 3)

board_edges[0] = (board_edges[0][0] - 15, board_edges[0][1] - 15)
board_edges[1] = (board_edges[1][0] + 15, board_edges[1][1] - 15)
board_edges[2] = (board_edges[2][0] + 15, board_edges[2][1] + 15)
board_edges[3] = (board_edges[3][0] - 15, board_edges[3][1] + 15)

for e in board_edges:
    cv2.circle(gray, (int(e[0]), int(e[1])), 5, (255, 0, 255), -1)
    cv2.circle(img_orig, (int(e[0]), int(e[1])), 20, (255, 0, 255), -1)

warped = four_point_transform(img_orig, board_edges)
ipyplot.plot_images(
    [Image.fromarray(img_orig), T.ToPILImage()(img), Image.fromarray(gray), Image.fromarray(warped)], img_width=450
)